# The plan, step by step

This notebook has one cell for each step of [PLAN.md](https://github.com/gbirhanu/ondevice-hallu/blob/main/PLAN.md)
that needs the GPU. The step numbers are the same as in the plan.

**How to use it:** choose a T4 GPU (Runtime > Change runtime type; steps 9, 13, 14b and 8e need only a CPU), run the two setup
cells, then run only the cell of the step you are doing. Results go to `MyDrive/ondevice_hallu/work`, next to the earlier results.
Every command skips work that is already saved, so after a disconnect, run the two setup cells and the same
step cell again: it continues where it stopped. Keep the browser tab open while a step runs.

| Step | What it does | Needs | Time on a free T4 |
|---|---|---|---|
| 5 | Test A at 68M: small-model answers against 7B answers | nothing | about 2.5 hours |
| 6 | Two more training runs (seeds 2 and 3) of the four main students | step 2 agreed | 1.2 + 5.5 hours |
| 7 | Two more training runs of test A at 17M | step 2 agreed | about 1 hour |
| — | The averaged tables (`results/summary.md`) | any of 5–7 | 15–30 minutes |
| 8b | **Added 13 Nov.** 17M: other learning rates (dev set only), then three runs with the chosen one | nothing | 1.7 + 1.1 hours |
| 8c | **Added 13 Nov.** A 32M student: learning rate (dev set only), then three runs | nothing | about 3.5 + 2.5 hours (estimate) |
| 8d | Tables for the added runs (`results/summary_added.md`) | 8b or 8c | 10–20 minutes |
| 8e | 8-bit files of the new students: size, accuracy and time | 8d | about 1 hour, **CPU only** |
| 9 | How many errors each set of answers holds | nothing | a few minutes, no GPU |
| 12 | Baseline: phone-sized chat models (0.5B, 1.5B) as the checker | nothing | about 30 minutes |
| 13 | The 8-bit files of our 68M and 17M checkers: size and accuracy | step 8 | about 2 hours, **CPU only** |
| 14 (fallback) | Time per answer on Colab's CPU, if the laptop timing is not possible | step 13 | about 45 minutes, **CPU only** |
| 14b | Accuracy of the 8-bit 68M with inputs cut to 1,024 tokens (exploratory) | step 14 | about 1.5 hours, **CPU only** |
| 17, 20 | Added here when we reach them | see PLAN.md | |

Everything is free: only the Colab GPU is used, no paid service.

In [ ]:
# Setup 1 of 2: Drive, code, data and libraries (about 3 minutes)
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, subprocess, shutil
from pathlib import Path
from IPython.display import Markdown, display

REPO = 'https://github.com/gbirhanu/ondevice-hallu.git'
CODE = Path('/content/ondevice_hallu')
WORK = Path('/content/drive/MyDrive/ondevice_hallu/work')
DATA = Path('/content/RAGTruth')

def sh(command):
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
    print(result.stdout[-3000:], flush=True)
    return result.returncode

if (CODE / '.git').exists():
    sh(['git', '-C', str(CODE), 'pull', '--ff-only'])
else:
    assert sh(['git', 'clone', REPO, str(CODE)]) == 0, f'Cannot clone {REPO}.'
if not DATA.exists():
    sh(['git', 'clone', '--depth', '1', 'https://github.com/ParticleMedia/RAGTruth.git', str(DATA)])
sh([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.57.6', 'accelerate', 'bitsandbytes', 'datasets',
    'onnx', 'onnxruntime==1.29.0', 'onnx-ir==1.0.0', 'psutil'])
try:                                     # optional: a Hugging Face token in the Secrets panel (key icon)
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass                                 # not needed for these steps
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU. That is fine for step 13 (CPU only); every other step needs Runtime > Change runtime type > T4 GPU.')

In [ ]:
# Setup 2 of 2: helpers used by every step
def run(*arguments):
    """Runs one run.py command and shows its output as it comes. Returns True when it worked."""
    command = [sys.executable, '-u', str(CODE / 'run.py'), *map(str, arguments),
               '--data-dir', str(DATA), '--work-dir', str(WORK)]
    process = subprocess.Popen(command, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                               env={**os.environ, 'TOKENIZERS_PARALLELISM': 'false'})
    for line in process.stdout:
        if not any(x in line for x in ('Loading checkpoint shards', 'Warning', 'Writing model shards', 'Loading weights')):
            print(line, end='', flush=True)
    return process.wait() == 0

def free_disk():
    """Deletes downloaded models from Colab's own disk (never from Drive)."""
    hub = Path(os.environ.get('HF_HOME', Path.home() / '.cache' / 'huggingface')) / 'hub'
    for d in hub.glob('models--*'):
        shutil.rmtree(d, ignore_errors=True)

def show(name):
    display(Markdown((WORK / 'results' / name).read_text()))

def kept(gens):
    """New answers kept per answering model (written by "generate")."""
    return {g: json.loads((WORK / 'results' / f'extra_{g}.json').read_text())['kept']
            for g in gens if (WORK / 'results' / f'extra_{g}.json').exists()}

TEACHER = 'ld_large'                                       # LettuceDetect-large, 396M
SMALL = ['qwen2.5-0.5b', 'qwen2.5-1.5b', 'smollm2-1.7b']   # phone-sized answering models
LARGE = ['qwen2.5-7b', 'olmo2-7b', 'mistral-7b-v0.3']      # 7B answering models (OLMo-2 did not run)
small, large = kept(SMALL), kept(LARGE)
N = min(sum(small.values()), sum(large.values())) if small and large else 0
SMALL_RUN = 'kd{size}' if sum(small.values()) <= N * 1.02 else 'kd{size}-n%d' % N   # same number of new answers

def train_and_score(name, size, *options, epochs=5):
    """Trains a student exactly like kd17m / kd68m (Ettin, 5 epochs, learning rate 2e-5), then scores it."""
    ok = run('train', '--name', name, '--base', f'jhu-clsp/ettin-encoder-{size}', '--epochs', epochs,
             '--lr', 2e-5, '--teacher', TEACHER, *options)
    return ok and run('score', '--name', name)

# The four main students as trained on 18 Oct: (size, epochs, options). Step 6 repeats them with other seeds.
MAIN = {'hard17m-8ep': ('17m', 8, []), 'kd17m': ('17m', 5, ['--teacher', TEACHER, '--extra', *small]),
        'hard68m-8ep': ('68m', 8, []), 'kd68m': ('68m', 5, ['--teacher', TEACHER, '--extra', *small])}

def again(base_run, seed):
    """Trains one of MAIN again with another seed (the run is named <run>-s<seed>), then scores it."""
    size, epochs, options = MAIN[base_run]
    name = f'{base_run}-s{seed}'
    ok = run('train', '--name', name, '--base', f'jhu-clsp/ettin-encoder-{size}', '--epochs', epochs,
             '--lr', 2e-5, '--seed', seed, *options)
    assert ok and run('score', '--name', name), f'{name} failed; see the message above.'

# Added 13 November (steps 8b-8e): other learning rates, tried on the dev set only, and the 32M size.
LRS = {'17m': [5e-5, 1e-4, 2e-4], '32m': [5e-5, 1e-4]}     # tried next to 2e-5, the learning rate in use
STUDENTS = {'hard{size}-8ep': (8, []), 'kd{size}': (5, ['--teacher', TEACHER, '--extra', *small])}

def lr_name(student, lr):
    """kd17m trained with learning rate 5e-5 is the run kd17m-lr5e5; with 2e-5 it is kd17m itself."""
    return student if lr == 2e-5 else f'{student}-lr{lr:.0e}'.replace('e-0', 'e')

def lr_test(size):
    """Seed 42 of both students of one size, once per learning rate. Training only: no test set is scored
    (rule R1). Then "lrpick" keeps 2e-5 unless another value is at least 2 dev points better, and saves the choice."""
    for student, (epochs, options) in STUDENTS.items():
        student = student.format(size=size)
        rates = [2e-5, *LRS[size]]
        names = [lr_name(student, lr) for lr in rates]
        for name, lr in zip(names, rates):
            assert run('train', '--name', name, '--base', f'jhu-clsp/ettin-encoder-{size}', '--epochs', epochs,
                       '--lr', lr, *options), f'{name} failed; see the message above.'
        assert run('lrpick', '--names', *names), 'The choice failed; see the message above.'
    for student in STUDENTS:
        show(f'lrpick_{student.format(size=size)}.md')

def pick(student):
    """The run "lrpick" chose for this student and its learning rate, or (None, None) before the choice."""
    f = WORK / 'results' / f'lrpick_{student}.json'
    c = json.loads(f.read_text()) if f.exists() else {}
    return c.get('chosen'), c.get('chosen_lr')

def three_runs(size):
    """The chosen run of each student is scored, then trained again with seeds 2 and 3 and scored."""
    for student, (epochs, options) in STUDENTS.items():
        name, lr = pick(student.format(size=size))
        assert name, f'Run the learning-rate cell of the {size.upper()} students first.'
        assert run('score', '--name', name), f'Scoring {name} failed; see the message above.'
        for seed in (2, 3):
            ok = run('train', '--name', f'{name}-s{seed}', '--base', f'jhu-clsp/ettin-encoder-{size}',
                     '--epochs', epochs, '--lr', lr, '--seed', seed, *options)
            assert ok and run('score', '--name', f'{name}-s{seed}'), f'{name}-s{seed} failed; see the message above.'
        print(f'--> {name}: three runs trained and scored.')

def three_scored(name):
    return bool(name) and all((WORK / 'results' / f'{n}_score.json').exists() for n in (name, f'{name}-s2', f'{name}-s3'))

print('new answers kept, small models:', small)
print('new answers kept, 7B models:   ', large, '-> each test A student gets', N, 'new answers')
need = ['results/kd68m_training.json', 'results/kd17m-n4509_score.json', 'results/kdlarge17m_score.json',
        'preds/ld_large_train.json.gz', 'preds/ld_large_test.json.gz', 'preds/psilo_gold.json.gz',
        *[f'preds/{TEACHER}_extra_{g}.json.gz' for g in [*small, *large]]]
missing = [n for n in need if not (WORK / n).exists()]
print('Earlier results this notebook needs, missing:', missing or 'none (good)')

## Step 5 — Test A at 68M

Does it matter **who writes** the new answers the student learns from? At 17M (19 Oct), the student trained on
4,509 answers from the small models (0.5–1.7B) beat the one trained on 4,509 answers from 7B models by 2.5
points on PsiloQA. This step trains the same pair at 68M:

- `kd68m-n4509`: 4,509 new answers from the small models;
- `kdlarge68m`: 4,509 new answers from Qwen2.5-7B and Mistral-7B v0.3.

Everything else is the same as `kd68m`. Each run takes about 70 minutes; then both are scored on RAGTruth and
PsiloQA, and the table is written to `results/compare_step5.md` (the older `compare.md` files are not touched).
The two new models take about 0.3 GB on Drive.

In [ ]:
# Step 5: test A at 68M (about 2.5 hours)
size = '68m'
small_run = SMALL_RUN.format(size=size)                                  # kd68m-n4509
assert train_and_score(small_run, size, '--extra', *small, '--extra-max', N), 'Training failed; see the message above.'
assert train_and_score(f'kdlarge{size}', size, '--extra', *large, '--extra-max', N), 'Training failed; see the message above.'
if not run('psilo'):                                                     # scores only runs not scored yet
    print('--> PsiloQA scoring failed (see above); the table below then has RAGTruth columns only.')
pairs = [f'{SMALL_RUN.format(size="17m")}:kdlarge17m',                    # test A at 17M (19 Oct), for reference
         f'{small_run}:kdlarge{size}',                                    # test A at 68M: the new result
         f'kd{size}:{small_run}']                                         # what 4,509 instead of 6,719 new answers cost
run('compare', '--pairs', *pairs, '--out', 'compare_step5.md')
show('compare_step5.md')

**How to read it.** Each cell is *span F1 of A → span F1 of B (B minus A, 95% interval)*.

- **Row 2 (`kd68m-n4509 → kdlarge68m`) is the new result.** A negative number means the student that learned
  from **small** models' answers is better. At 17M (row 1) the PsiloQA difference was −2.50 [−3.17, −1.79] and
  the RAGTruth difference −1.02 [−2.69, 0.55] (no clear difference). If row 2 points the same way, the finding
  holds at both sizes. It is still one training run per student; steps 6 and 7 add two more seeds.
- **Row 3** shows how much the 68M student loses with 4,509 instead of 6,719 new answers.
- When it is done, tell Claude "step 5 done": Claude reads the table from Drive and updates the status log.

## Step 6 — Two more training runs of the four main students

**Run this only after `EVALUATION.md` (step 2) is agreed and pushed.** The four students of 18 October are
trained again with seeds 2 and 3, with exactly the same recipe; `run.py` refuses to start a run whose recipe
differs from the first run's. A run named `kd17m-s2` is `kd17m` trained with seed 2.

| Student | What it learns from | One run on a T4 |
|---|---|---|
| `hard17m-8ep`, `hard68m-8ep` | people's labels only, 8 epochs | 17 / 83 minutes |
| `kd17m`, `kd68m` | people's labels + the teacher, and 6,719 new answers from small models | 16 / 77 minutes |

The work is split into two cells so that each fits in one Colab session: the 17M runs (about 1.2 hours) and the
68M runs (about 5.5 hours; if the session ends, run the setup cells and the same cell again: finished runs are
skipped in seconds). The 8 new models take about 0.7 GB on Drive.

In [ ]:
# Step 6, 17M part: 4 runs, about 1.2 hours
for seed in (2, 3):
    for base_run in ('hard17m-8ep', 'kd17m'):
        again(base_run, seed)

In [ ]:
# Step 6, 68M part: 4 runs, about 5.5 hours (it can be spread over two sessions)
for seed in (2, 3):
    for base_run in ('hard68m-8ep', 'kd68m'):
        again(base_run, seed)

## Step 7 — Two more training runs of test A at 17M

**Also only after step 2.** `kd17m-n4509` (4,509 new answers from small models) and `kdlarge17m` (4,509 from
7B models), each with seeds 2 and 3: 4 runs, about 1 hour. It needs only the two setup cells.

In [ ]:
# Step 7: test A at 17M with seeds 2 and 3 (about 1 hour)
for seed in (2, 3):
    for name, options in [(SMALL_RUN.format(size='17m'), ['--extra', *small, '--extra-max', N]),
                          ('kdlarge17m', ['--extra', *large, '--extra-max', N])]:
        assert train_and_score(f'{name}-s{seed}', '17m', *options, '--seed', seed), 'Training failed; see above.'

## The averaged tables (after steps 5, 6 and 7)

Scores the new runs on PsiloQA and writes `results/summary.md`: every checker as the mean ± spread of its
training runs, and the differences the success criteria need, run by run, with "Counts?" applying rule R3 (all
three runs point the same way and the 95% interval excludes 0). It can be run at any time; runs that are missing
are left out. It takes 15–30 minutes, mostly scoring the new runs on PsiloQA (2–3 minutes each). Then tell Claude
"summary done" (step 8).

In [ ]:
# The averaged tables: results/summary.md and results/summary.json
if not run('psilo'):
    print('--> PsiloQA scoring failed (see above); the summary then has fewer PsiloQA numbers.')
assert run('summary'), 'The summary failed; see the message above.'
show('summary.md')

## Steps 8b–8e — added on 13 November 2026

These steps were **added after the results of 31 October were seen** (EVALUATION.md, section 9), so their results are
reported as added later. Success criteria S1–S4 stay judged on the runs made with the original recipe.

**Why.** (1) With learning rate 2e-5 the 17M students were still improving on the dev set at their last epoch (the
best epoch of `kd17m` was the last one in all three runs), and they vary by ± 3.8 points between runs. (2) The 68M
needs 1.66 s per answer on one laptop core and the 17M is weak; the 32M size lies between them.

**The rule, fixed before any of these runs.** For each student, seed 42 is trained once per learning rate. Only the
dev set is looked at (span F1 at 0.5 of the best epoch); no test set is scored. The student keeps 2e-5 unless
another value is **at least 2.0 dev points better**; then the best one is chosen. `run.py lrpick` applies the rule
and saves the choice in `results/lrpick_<student>.json`; a saved choice is not changed. The chosen run is then
repeated with seeds 2 and 3. Everything else (epochs, α, new answers, max length, batch size) stays as it was.

## Step 8b — The 17M students with other learning rates

**Part 1 (about 1.7 hours, GPU):** `hard17m-8ep` and `kd17m`, seed 42, with learning rates 5e-5, 1e-4 and 2e-4
(6 runs of 15–17 minutes; the 2e-5 runs exist already and are skipped). The cell ends with two small tables that
say which learning rate each student keeps.

**Part 2 (about 1.1 hours, GPU):** the chosen runs with seeds 2 and 3 (4 runs). If both students keep 2e-5,
nothing new is trained and the cell ends in a few seconds.

In [ ]:
# Step 8b, part 1: the 17M students with learning rates 5e-5, 1e-4 and 2e-4 (dev set only, about 1.7 hours)
lr_test('17m')

In [ ]:
# Step 8b, part 2: three runs of the chosen 17M students (about 1.1 hours; seconds if 2e-5 stays)
three_runs('17m')

## Step 8c — A 32M student

The same two students at the 32M size (`jhu-clsp/ettin-encoder-32m`): `hard32m-8ep` (people's labels only, 8 epochs)
and `kd32m` (people's labels + the teacher, and the same 6,719 new answers from small models, 5 epochs). The
published model of this size is TinyLettuce-32M (47.86 span F1 on RAGTruth).

**Part 1 (about 3.5 hours, GPU; an estimate, the first run shows the real time):** seed 42 with learning rates
2e-5, 5e-5 and 1e-4 (6 runs), then the choice by the same rule.

**Part 2 (about 2.5 hours, GPU):** the chosen runs with seeds 2 and 3 (4 runs).

If a session ends, run the two setup cells and the same cell again: finished runs are skipped in seconds.

In [ ]:
# Step 8c, part 1: the 32M students with learning rates 2e-5, 5e-5 and 1e-4 (dev set only, about 3.5 hours)
lr_test('32m')

In [ ]:
# Step 8c, part 2: three runs of the chosen 32M students (about 2.5 hours)
three_runs('32m')

## Step 8d — Tables for the added runs

Scores the new runs on PsiloQA (and the published TinyLettuce-32M, once) and writes `results/summary_added.md`.
`results/summary.md`, the table of the planned comparisons, is not touched. Run it after step 8b, and again after
step 8c: sizes that are not finished are left out. It takes 10–20 minutes on a GPU runtime. Then tell Claude
"step 8d done".

In [ ]:
# Step 8d: results/summary_added.md, the comparisons of the added runs (10-20 minutes)
if not run('psilo'):
    print('--> PsiloQA scoring failed (see above); the table then has fewer PsiloQA numbers.')
pairs = ['ld_base:kd68m']                                         # our 68M against the published 150M
(h17, _), (k17, _) = pick('hard17m-8ep'), pick('kd17m')
if three_scored(h17) and three_scored(k17):
    pairs += [f'{h17}:{k17}', f'tiny17m:{k17}', f'tiny68m:{k17}']  # what the teacher adds; against the published 17M and 68M
    pairs += [f'kd17m:{k17}'] if k17 != 'kd17m' else []            # what the new learning rate changed
    pairs += [f'hard17m-8ep:{h17}'] if h17 != 'hard17m-8ep' else []
(h32, _), (k32, _) = pick('hard32m-8ep'), pick('kd32m')
if three_scored(h32) and three_scored(k32):
    pairs += [f'{h32}:{k32}', f'tiny32m:{k32}', f'tiny68m:{k32}', f'{k32}:kd68m']
print('Comparisons:', pairs)
assert run('summary', '--pairs', *pairs, '--out', 'summary_added.md'), 'The summary failed; see the message above.'
show('summary_added.md')

**How to read it.** The first table lists every checker; the second the added comparisons, run by run, with the
same rule R3 in "Counts?". Rows to look at: `hard…→kd…` (what the teacher and the new answers add at that size),
`tiny32m→kd32m…` and `tiny68m→kd32m…` (does our 32M beat the published model of its size, and reach the published
68M?), `kd17m→kd17m-lr…` (what the new learning rate changed), `ld_base→kd68m` (our 68M against the published 150M).

## Step 8e — 8-bit files of the new students: size, accuracy and time

For the chosen `kd32m` run, and the chosen `kd17m` run if its learning rate changed (seed-42 runs): the 8-bit file,
its accuracy on all 2,700 RAGTruth test answers and on PsiloQA, and the time per answer on Colab's CPU. **Run step
8d first** (it scores the full-precision models on PsiloQA). Use a **CPU** runtime; about 1 hour. The time that
counts for S5 is the one on your laptop: the commands are in PLAN.md, step 8e.

In [ ]:
# Step 8e: 8-bit files of the new students, their size, accuracy and time on Colab's CPU (CPU only, about 1 hour)
ONNX = Path('/content/onnx')                          # Colab's own disk, not Drive
names = [n for n in (pick('kd32m')[0], pick('kd17m')[0]) if n and n != 'kd17m']
assert names, 'No new student yet: run steps 8b and 8c first.'
for name in names:
    assert run('export', '--name', name, '--onnx-dir', ONNX), f'Export of {name} failed; see the message above.'
    assert run('check8bit', '--name', name, '--n', 2700, '--psilo', '--onnx-dir', ONNX), \
        f'The 8-bit check of {name} failed; see the message above.'
    assert run('bench', '--name', name, '--threads', 1, 2, '--where', 'colab', '--onnx-dir', ONNX), \
        f'Timing of {name} failed; see the message above.'

rows = ['| Checker | File size: full → 8-bit (MB) | RAGTruth: full → 8-bit, change [95%] | PsiloQA: change [95%] | '
        'Colab CPU, 8-bit, 1 thread: median / 90th percentile (ms) |', '|---|---|---|---|---|']
for name in [*names, 'kd68m', 'kd17m']:
    files = [WORK / 'results' / f'{name}_{x}.json' for x in ('export', 'int8_n2700', 'int8_psilo', 'bench_colab')]
    if not all(f.exists() for f in files[:2]):
        continue
    e, r = (json.loads(f.read_text()) for f in files[:2])
    q = json.loads(files[2].read_text())['groups']['all'] if files[2].exists() else None
    t = next((x for x in json.loads(files[3].read_text())['rows'] if x['weights'] == 'int8' and x['threads'] == 1),
             None) if files[3].exists() else None
    rows.append(f"| {name} | {e['fp32_mb']} → {e['int8_mb']} | {r['span_f1_full_precision']} → {r['span_f1_int8']}, "
                f"{r['change']:+.2f} {r['change_ci95']} | " + (f"{q['change']:+.2f} {q['change_ci95']}" if q else '')
                + ' | ' + (f"{t['median_ms']} / {t['p90_ms']}" if t else '') + ' |')
display(Markdown('\n'.join(rows)))

## Step 9 — How many errors does each set of answers hold?

For the new answers of every answering model (as our teacher marks them) and for RAGTruth's training answers by
answering model (as people and as the teacher mark them): the share of answers with at least one marked error, the
share of marked characters, and the answer length. **No GPU and no training**: a few minutes on any runtime.

In [ ]:
# Step 9: error counts of every set of answers (no GPU, a few minutes)
assert run('density'), 'The count failed; see the message above.'
show('density.md')

## Step 12 — Baseline: can the phone's own chat model check answers?

A phone that runs a 0.5B or 1.5B chat model could ask **that same model** to list the unsupported parts of an
answer, with no extra checker. This step measures how well that works, on the same 300 RAGTruth test answers on
which Qwen2.5-7B scored 11.1 (19 Oct). Our teacher and our students are scored on the same 300 answers, so all
numbers in the table are comparable. A reply the program cannot read counts as "marks nothing", which is what
an app would show.

In [ ]:
# Step 12: phone-sized chat models as the checker (about 30 minutes)
SELF = {'selfcheck05': 'qwen2.5-0.5b', 'selfcheck15': 'qwen2.5-1.5b'}
for name, model in SELF.items():
    assert run('llmlabel', '--name', name, '--local', model, '--on', 'test', '--limit', 300), 'Labelling failed.'
    free_disk()
if (WORK / 'llm' / 'qwen7b' / 'test').exists():   # the 7B labeller of 19 Oct; its replies are saved, so no GPU work
    run('llmlabel', '--name', 'qwen7b', '--local', 'qwen2.5-7b', '--on', 'test', '--limit', 300)

SIZE = {'selfcheck05': '0.5B', 'selfcheck15': '1.5B', 'qwen7b': '7B', 'ld_large': '396M', 'ld_base': '150M',
        'tiny17m': '17M', 'tiny68m': '68M', 'hard17m-8ep': '17M', 'kd17m': '17M', 'hard68m-8ep': '68M', 'kd68m': '68M'}
res = {n: json.loads((WORK / 'results' / f'llmlabel_{n}_test.json').read_text())
       for n in ['selfcheck05', 'selfcheck15', 'qwen7b'] if (WORK / 'results' / f'llmlabel_{n}_test.json').exists()}
ref = res['selfcheck15']['all_answers']
rows = ['| Checker | Size | Span F1 on the 300 answers [95%] | Minus Qwen2.5-1.5B [95%] | Replies it could not read |',
        '|---|---|---|---|---|']
for n, r in res.items():
    a = r['all_answers']['labeller']
    rows.append(f"| {r['llm_model']} (chat model, asked to list unsupported parts) | {SIZE[n]} | "
                f"{a['span_f1']} {a['ci95']} | | {r['unusable_replies']} of {r['answers']} |")
for n, c in ref['checkers'].items():
    rows.append(f"| {n} | {SIZE.get(n, '')} | {c['span_f1']} {c['ci95']} | "
                f"{c['span_f1'] - ref['labeller']['span_f1']:+.2f} {c['minus_labeller_ci95']} | |")
display(Markdown('\n'.join(rows)))

**How to read it.** If the 0.5B and 1.5B chat models score far below our 17M and 68M checkers (the last column
is positive with an interval above 0), the phone cannot simply ask its own model: a separate small checker is
needed, and that is the case for our work. Step 20 repeats this on our own test set. When it is done, tell
Claude "step 12 done".

## Step 13 — The 8-bit files and what they cost

The phone will run our checker as an 8-bit ONNX file. This step makes the 8-bit files of `kd68m` and `kd17m`
(the seed-42 runs) and scores them on **all** 2,700 RAGTruth test answers and the 4,000 PsiloQA answers, against
the same models in full precision. It also records the file sizes.

**It needs no GPU:** choose Runtime > Change runtime type > **CPU** (this saves your free GPU time), run the two
setup cells, then this cell. It takes about 2 hours, one answer at a time as on a phone. The ONNX files stay on
Colab's own disk, so Drive does not fill up; only small result files go to Drive. If the session ends, run the
setup cells and this cell again: finished parts are skipped.

In [ ]:
# Step 13: 8-bit files of kd68m and kd17m, their size and their accuracy (CPU only, about 2 hours)
ONNX = Path('/content/onnx')                          # Colab's own disk, not Drive
for name in ('kd68m', 'kd17m'):
    assert run('export', '--name', name, '--onnx-dir', ONNX), f'Export of {name} failed; see the message above.'
    assert run('check8bit', '--name', name, '--n', 2700, '--psilo', '--onnx-dir', ONNX), \
        f'The 8-bit check of {name} failed; see the message above.'

rows = ['| Checker | File size: full → 8-bit (MB) | RAGTruth: full → 8-bit, change [95%] | PsiloQA: full → 8-bit, change [95%] |',
        '|---|---|---|---|']
for name in ('kd68m', 'kd17m'):
    e = json.loads((WORK / 'results' / f'{name}_export.json').read_text())
    r = json.loads((WORK / 'results' / f'{name}_int8_n2700.json').read_text())
    q = json.loads((WORK / 'results' / f'{name}_int8_psilo.json').read_text())['groups']['all']
    rows.append(f"| {name} | {e['fp32_mb']} → {e['int8_mb']} | {r['span_f1_full_precision']} → {r['span_f1_int8']}, "
                f"{r['change']:+.2f} {r['change_ci95']} | {q['span_f1_full_precision']} → {q['span_f1_int8']}, "
                f"{q['change']:+.2f} {q['change_ci95']} |")
display(Markdown('\n'.join(rows)))

**How to read it.** Success criterion S5 needs the 8-bit `kd68m` to lose at most 1 point on RAGTruth (a change of
−1.00 or better) and its 8-bit file to be at most 150 MB. When it is done, tell Claude "step 13 done". Step 14
then times the 8-bit file on your laptop.

## Step 14 (fallback) — Time per answer on Colab's CPU

The real measure of step 14 is on your own laptop (see PLAN.md, step 14). If that is not possible, this cell
times the 8-bit and the full-precision files here instead: one answer at a time, 300 fixed RAGTruth test answers,
with 1 and 2 threads. Use a **CPU** runtime; it takes about 45 minutes.

In [ ]:
# Step 14 (fallback): time per answer on Colab's CPU (about 45 minutes)
ONNX = Path('/content/onnx')                          # remade here if this is a new session
for name in ('kd68m', 'kd17m'):
    assert run('bench', '--name', name, '--threads', 1, 2, '--where', 'colab', '--onnx-dir', ONNX), \
        f'Timing of {name} failed; see the message above.'
rows = ['| Checker | Weights | Threads | Median (ms) | 90th percentile (ms) | Memory added (MB) |',
        '|---|---|---|---|---|---|']
for name in ('kd68m', 'kd17m'):
    b = json.loads((WORK / 'results' / f'{name}_bench_colab.json').read_text())
    for r in b['rows']:
        rows.append(f"| {name} | {r['weights']} | {r['threads']} | {r['median_ms']} | {r['p90_ms']} | {r.get('model_mem_mb', '')} |")
display(Markdown('\n'.join(rows)))
print('CPU:', b['cpu'], '| cores:', b['cores'])

## Step 14b — The 8-bit 68M with shorter inputs (exploratory)

On one laptop core the 8-bit 68M needed 1.66 s per answer (the target was 1 s). Three faster variants were planned:

- **(a) the `mix` recipe** (8-bit arithmetic in the layers where it looked safe): run on 2 November, and it **fails**:
  span F1 on RAGTruth falls from 56.02 to 34.59 (−21.43 [−24.52, −18.21]). Closed; variant (c), which also uses
  `mix`, is dropped with it.
- **(b) inputs cut to 1,024 tokens** (the passages are shortened, the answer is kept): on the laptop this took
  1.23 s per answer. Its **accuracy** is still to be measured, and this cell does it, on all 2,700 RAGTruth test answers.

Use a **CPU** runtime. It takes about 1.5 hours; if the session ends, run the setup cells and this cell again. The
variant was chosen after seeing step 14, so it is reported as exploratory.

In [ ]:
# Step 14b: accuracy of the 8-bit kd68m with inputs cut to 1,024 tokens (CPU only, about 1.5 hours)
ONNX = Path('/content/onnx')                          # Colab's own disk, not Drive
name = 'kd68m'
assert run('check8bit', '--name', name, '--n', 2700, '--onnx-dir', ONNX, '--recipe', 'w8', '--max-length', 1024), \
    'The check failed; see the message above.'                       # makes the ONNX files itself when they are missing

rows = ['| 8-bit kd68m | Inputs up to | RAGTruth: full → 8-bit, change [95%] | PsiloQA: change [95%] |', '|---|---|---|---|']
for label, v, length in [('planned (w8)', '', 4096), ('(a) mix: fails, closed', '_mix', 4096), ('(b) w8', '_len1024', 1024)]:
    f, q = WORK / 'results' / f'{name}_int8{v}_n2700.json', WORK / 'results' / f'{name}_int8{v}_psilo.json'
    if not f.exists():
        continue
    r = json.loads(f.read_text())
    a = json.loads(q.read_text())['groups']['all'] if q.exists() else None
    rows.append(f"| {label} | {length} | {r['span_f1_full_precision']} → {r['span_f1_int8']}, {r['change']:+.2f} "
                f"{r['change_ci95']} | " + (f"{a['change']:+.2f} {a['change_ci95']}" if a else '') + ' |')
display(Markdown('\n'.join(rows)))

## Next steps

The cells for steps 17 and 20 are added here when we reach them (step 10 was dropped; step 11 is optional).